### final_model_train_predict

In [ ]:
#!/usr/bin/env python3
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Import required model components
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import json
import joblib

print("Poly3+Ridge (Fabric) + SVR (Style) Models - Full Training and Prediction Pipeline")


# ==================== Part 1: Model training ====================


def load_training_data():
    """Load the training data."""
    try:
        # Use the filename directly
        excel_path = 'style_fabric_preference.xlsx'

        if not os.path.exists(excel_path):
            raise FileNotFoundError(f"File not found: {excel_path}. Please make sure it is in the current directory.")

        # Read the style training data
        df_style = pd.read_excel(excel_path, sheet_name='style_data')
        df_style = df_style.iloc[:, :5]
        df_style.columns = ['id', 'style', 'preference_mean', 'search_index', 'info_index']

        # Read the fabric training data
        df_fabric = pd.read_excel(excel_path, sheet_name='fabric_data')
        df_fabric = df_fabric.iloc[:, :5]
        df_fabric.columns = ['id', 'fabric', 'preference_mean', 'search_index', 'info_index']

        # Preprocessing function
        def preprocess(df):
            df = df.copy()
            # Convert key columns to numeric
            for col in ['search_index', 'info_index', 'preference_mean']:
                df[col] = pd.to_numeric(df[col], errors='coerce')

            # Drop rows with missing values in key columns
            df = df.dropna(subset=['search_index', 'info_index', 'preference_mean'])
            # Keep only rows where preference_mean is in [0, 0.3]
            df = df[(df['preference_mean'] >= 0) & (df['preference_mean'] <= 0.3)]

            # Log-transform the independent variables (add 1 to avoid log(0))
            df['ln_search'] = np.log(df['search_index'].astype(float) + 1)
            df['ln_info'] = np.log(df['info_index'].astype(float) + 1)

            return df

        df_style = preprocess(df_style)
        df_fabric = preprocess(df_fabric)

        print(f"Training data loaded: style={len(df_style)}, fabric={len(df_fabric)}")

        return df_style, df_fabric

    except Exception as e:
        print(f"Failed to load training data: {e}")
        import traceback
        traceback.print_exc()
        return None, None


# ==================== Style model: SVR (both) ====================


def create_svr_model(C=1.0, epsilon=0.1, gamma='scale'):
    """Create an SVR model (for style prediction)."""
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('svr', SVR(kernel='rbf', C=C, epsilon=epsilon, gamma=gamma))
    ])
    return model


def train_svr_model(X, y, dataset_name):
    """Train the SVR model with hyperparameter tuning."""
    param_grid = {
        'svr__C': [0.1, 1.0, 10.0],
        'svr__epsilon': [0.01, 0.1, 0.5],
        'svr__gamma': ['scale', 'auto', 0.1, 1.0]
    }

    grid_search = GridSearchCV(
        create_svr_model(), param_grid, cv=5, scoring='r2',
        n_jobs=-1, verbose=0
    )
    grid_search.fit(X, y)

    best_model = grid_search.best_estimator_
    best_model.fit(X, y)

    y_pred = best_model.predict(X)
    train_r2 = r2_score(y, y_pred)
    train_rmse = np.sqrt(mean_squared_error(y, y_pred))

    best_params = grid_search.best_params_
    best_C = best_params['svr__C']
    best_epsilon = best_params['svr__epsilon']
    best_gamma = best_params['svr__gamma']

    print(f"[{dataset_name}] SVR  best: C={best_C}, eps={best_epsilon}, gamma={best_gamma} "
          f"| Train R2={train_r2:.4f}, RMSE={train_rmse:.6f}")

    return best_model, best_C, best_epsilon, best_gamma


# ==================== Fabric model: Poly3+Ridge (both) ====================


def create_poly3_ridge_model(alpha=1.0):
    """Create a Poly3+Ridge model (for fabric prediction)."""
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=3, include_bias=False, interaction_only=False)),
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=alpha, random_state=42))
    ])
    return model


def train_poly3_ridge_model(X, y, dataset_name):
    """Train the Poly3+Ridge model with hyperparameter tuning."""
    param_grid = {
        'ridge__alpha': [0.1, 1.0, 10.0, 100.0]
    }

    grid_search = GridSearchCV(
        create_poly3_ridge_model(), param_grid, cv=5, scoring='r2',
        n_jobs=-1, verbose=0
    )
    grid_search.fit(X, y)

    best_model = grid_search.best_estimator_
    best_model.fit(X, y)

    y_pred = best_model.predict(X)
    train_r2 = r2_score(y, y_pred)
    train_rmse = np.sqrt(mean_squared_error(y, y_pred))

    best_alpha = grid_search.best_params_['ridge__alpha']

    print(f"[{dataset_name}] Poly3+Ridge  best: alpha={best_alpha} "
          f"| Train R2={train_r2:.4f}, RMSE={train_rmse:.6f}")

    return best_model, best_alpha


def train_models():
    """Train both the style and fabric models."""
    df_style_train, df_fabric_train = load_training_data()
    if df_style_train is None or df_fabric_train is None:
        print("Failed to load training data. Exiting.")
        return None, None, None, None

    X_style = df_style_train[['ln_search', 'ln_info']].values
    y_style = df_style_train['preference_mean'].values
    X_fabric = df_fabric_train[['ln_search', 'ln_info']].values
    y_fabric = df_fabric_train['preference_mean'].values

    model_style, best_C, best_epsilon, best_gamma = train_svr_model(X_style, y_style, "Style")
    model_fabric, best_alpha = train_poly3_ridge_model(X_fabric, y_fabric, "Fabric")

    model_params = {
        'style_model': {
            'type': 'SVR',
            'kernel': 'rbf',
            'C': best_C,
            'epsilon': best_epsilon,
            'gamma': best_gamma
        },
        'fabric_model': {
            'type': 'Poly3+Ridge',
            'degree': 3,
            'alpha': best_alpha
        }
    }

    return model_style, model_fabric, model_params, (df_style_train, df_fabric_train)


def save_training_results(model_style, model_fabric, model_params):
    """Save the trained models and their parameters."""
    output_dir = 'poly3_ridge_svr_results'
    os.makedirs(output_dir, exist_ok=True)

    joblib.dump(model_style, os.path.join(output_dir, 'svr_style_model.pkl'))
    joblib.dump(model_fabric, os.path.join(output_dir, 'poly3_ridge_fabric_model.pkl'))

    with open(os.path.join(output_dir, 'model_info.json'), 'w', encoding='utf-8') as f:
        json.dump(model_params, f, ensure_ascii=False, indent=2)

    print(f"Models and parameters saved to '{output_dir}/'")


# ==================== Part 2: Test-set prediction ====================


def load_test_data():
    """Load the test data."""
    try:
        excel_path = 'style_fabric_preference.xlsx'

        if not os.path.exists(excel_path):
            raise FileNotFoundError(f"File not found: {excel_path}")

        df_style_test = pd.read_excel(excel_path, sheet_name='style_data')
        df_fabric_test = pd.read_excel(excel_path, sheet_name='fabric_data')

        def preprocess_test(df):
            df = df.copy()
            df['search_index'] = pd.to_numeric(df['search_index'], errors='coerce')
            df['info_index'] = pd.to_numeric(df['info_index'], errors='coerce')
            df = df.dropna(subset=['search_index', 'info_index'])

            df['ln_search'] = np.log(df['search_index'].astype(float) + 1)
            df['ln_info'] = np.log(df['info_index'].astype(float) + 1)

            return df

        df_style_test = preprocess_test(df_style_test)
        df_fabric_test = preprocess_test(df_fabric_test)

        print(f"Test data loaded: style={len(df_style_test)}, fabric={len(df_fabric_test)}")

        return df_style_test, df_fabric_test

    except Exception as e:
        print(f"Failed to load test data: {e}")
        import traceback
        traceback.print_exc()
        return None, None


def load_trained_models_for_prediction():
    """Load the trained models for prediction."""
    model_dir = 'poly3_ridge_svr_results'
    style_model_path = os.path.join(model_dir, 'svr_style_model.pkl')
    fabric_model_path = os.path.join(model_dir, 'poly3_ridge_fabric_model.pkl')

    if os.path.exists(style_model_path) and os.path.exists(fabric_model_path):
        model_style = joblib.load(style_model_path)
        model_fabric = joblib.load(fabric_model_path)
        print(f"Trained models loaded from '{model_dir}/'")
        return model_style, model_fabric
    else:
        print("No trained models found. Please run the training stage first.")
        return None, None


def make_predictions(model_style, model_fabric, df_style_test, df_fabric_test):
    """Run predictions."""
    X_style = df_style_test[['ln_search', 'ln_info']].values
    style_predictions = model_style.predict(X_style)
    df_style_test['preference_pred'] = style_predictions

    X_fabric = df_fabric_test[['ln_search', 'ln_info']].values
    fabric_predictions = model_fabric.predict(X_fabric)
    df_fabric_test['preference_pred'] = fabric_predictions

    print(f"Style  predictions: mean={style_predictions.mean():.6f}, "
          f"std={style_predictions.std():.6f}, "
          f"range=[{style_predictions.min():.6f}, {style_predictions.max():.6f}]")
    print(f"Fabric predictions: mean={fabric_predictions.mean():.6f}, "
          f"std={fabric_predictions.std():.6f}, "
          f"range=[{fabric_predictions.min():.6f}, {fabric_predictions.max():.6f}]")

    return df_style_test, df_fabric_test


def save_predictions(df_style, df_fabric):
    """Save the prediction results."""
    output_dir = 'poly3_ridge_svr_predictions'
    os.makedirs(output_dir, exist_ok=True)

    df_style[['style', 'province', 'search_index', 'info_index', 'preference_pred']] \
        .sort_values(['style', 'province']) \
        .to_csv(os.path.join(output_dir, 'style_predictions.csv'),
                index=False, encoding='utf-8-sig')

    df_fabric[['fabric', 'province', 'search_index', 'info_index', 'preference_pred']] \
        .sort_values(['fabric', 'province']) \
        .to_csv(os.path.join(output_dir, 'fabric_predictions.csv'),
                index=False, encoding='utf-8-sig')

    style_summary = df_style.groupby('style')['preference_pred'] \
        .agg(['mean', 'std', 'min', 'max', 'count']) \
        .round(6).sort_values('mean', ascending=False)
    style_summary.to_csv(os.path.join(output_dir, 'style_summary.csv'),
                         encoding='utf-8-sig')

    fabric_summary = df_fabric.groupby('fabric')['preference_pred'] \
        .agg(['mean', 'std', 'min', 'max', 'count']) \
        .round(6).sort_values('mean', ascending=False)
    fabric_summary.to_csv(os.path.join(output_dir, 'fabric_summary.csv'),
                          encoding='utf-8-sig')

    province_style = df_style.groupby('province')['preference_pred'] \
        .agg(['mean', 'std', 'count']) \
        .round(6).sort_values('mean', ascending=False)
    province_style.to_csv(os.path.join(output_dir, 'province_summary.csv'),
                          encoding='utf-8-sig')

    print(f"Saved 5 files to '{output_dir}/': "
          f"style/fabric predictions, style/fabric summaries, province summary")

    print("\nStyle Top 10:")
    print(style_summary.head(10).to_string())

    print("\nFabric Top 10:")
    print(fabric_summary.head(10).to_string())


# ==================== Part 3: Main program ====================


def main():
    """Main function - run the complete pipeline directly."""
    print("Starting pipeline: training + prediction")

    model_style, model_fabric, model_params, train_data = train_models()
    if model_style is None or model_fabric is None:
        print("Model training failed. Exiting.")
        return

    save_training_results(model_style, model_fabric, model_params)

    df_style_test, df_fabric_test = load_test_data()
    if df_style_test is None or df_fabric_test is None:
        print("Failed to load test data. Exiting.")
        return

    df_style_result, df_fabric_result = make_predictions(
        model_style, model_fabric, df_style_test, df_fabric_test
    )

    save_predictions(df_style_result, df_fabric_result)

    print("\nPipeline complete.")
    print(f"Style model:  SVR (C={model_params['style_model']['C']}, "
          f"epsilon={model_params['style_model']['epsilon']})")
    print(f"Fabric model: Poly3+Ridge (alpha={model_params['fabric_model']['alpha']})")


if __name__ == "__main__":
    main()